<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/09%20-%20Generative%20Models/09b_goal_directed_generation.ipynb)

# 09b · 有目标的生成：把生成模型和预测模型接起来

> ai4chem 第 5 章之后用 `REINVENT` 做强化学习定向生成。这里我们用更简单、更容易看懂的**微调**（fine-tuning）做同一件事，并且把第二、四部分的工具（随机森林、适用域）全都用上。

08a 的模型会写"像 CEP 的分子"，09a 的模型给了分子连续坐标。但**我们真正想要的是高效率的分子**。这一节把两件事接起来：

1. 先要有一个"裁判"：PCE 预测模型
2. 微调：让生成模型偏向高 PCE 的分子
3. 效果如何？
4. ⚠️ 陷阱一：模型欺骗
5. ⚠️ 陷阱二：能不能合成？
6. 一个诚实的检查清单

### 这个 notebook 怎么用

| 标记 | 意思 | 你要做什么 |
|---|---|---|
| 📘 | 讲解 | 读一读 |
| 🤔 | 先猜一猜 | **运行之前**先想想会输出什么 |
| ▶️ | 运行看看 | 选中代码格，按 `Shift + Enter` |
| ✏️ | 改一改 | 改代码里标了 `# ✏️` 的地方，再运行 |
| 🏋️ | 练习 | 自己写代码；参考答案在 `solutions/` 文件夹 |
| 💡 | 小贴士 | 有用但不是必须记住的东西 |


💡 这个 notebook 需要 `data/cep_rnn.pt`（08a 用的预训练模型）。随机森林训练约 1 分钟，微调约 1 分钟。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "09 - Generative Models"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import Draw, rdFingerprintGenerator
from rdkit.Chem.rdchem import RWMol
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

RDLogger.DisableLog("rdApp.*")
torch.manual_seed(0)
np.random.seed(0)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
import os
import pandas as pd

# 本地运行：数据在 data/ 文件夹里
# Colab 运行：如果找不到文件，会弹出上传按钮，选择 cep.csv 即可
DATA_DIR = "data"
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    DATA_DIR = "."
if not os.path.exists(os.path.join(DATA_DIR, "cep.csv")):
    try:
        from google.colab import files
        print("请上传 cep.csv（如果要用预训练模型，把 cep_rnn.pt 也一起上传）")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 cep.csv，请把它放在 data/ 文件夹或和 notebook 同一个文件夹里")

df = pd.read_csv(os.path.join(DATA_DIR, "cep.csv"))
print("读取成功！一共", len(df), "个分子")
df.head()

---
## 1. 先要有一个"裁判"

📘 要生成"PCE 高"的分子，首先得能**判断**一个分子的 PCE。CEP 数据集里的 PCE 来自 DFT 计算，算一个分子要很久——没法给生成模型当即时反馈。

所以第一步是训练一个快速的**代理模型**（surrogate model）：用第二部分的老办法，Morgan 指纹 + 随机森林。

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in df["smiles"]]
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fingerprints = np.array([fp_gen.GetFingerprintAsNumPy(m) for m in mols], dtype=np.uint8)
y = df["PCE"].values

train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(fingerprints[train_idx], y[train_idx])

pred_test = rf.predict(fingerprints[test_idx])
rmse = np.sqrt(np.mean((pred_test - y[test_idx]) ** 2))
r2 = 1 - np.sum((pred_test - y[test_idx]) ** 2) / np.sum((y[test_idx] - y[test_idx].mean()) ** 2)
print(f"随机森林代理模型: 测试 RMSE = {rmse:.2f}%,  R² = {r2:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(y[test_idx], pred_test, s=3, alpha=0.3)
axes[0].plot([-7, 12], [-7, 12], "k--", lw=1)
axes[0].set_xlabel("DFT PCE (%)"); axes[0].set_ylabel("predicted PCE (%)")
axes[0].set_title("surrogate model on test set")
axes[1].hist(y, bins=60)
axes[1].set_xlabel("PCE (%)"); axes[1].set_ylabel("count"); axes[1].set_title("PCE distribution in CEP")
plt.tight_layout(); plt.show()
print("PCE: 平均 %.2f%%, 中位数 %.2f%%, 最大 %.2f%%" % (y.mean(), np.median(y), y.max()))

📘 R² 约 0.74。不算特别准，但足够当一个**筛选**用的裁判了。

⚠️ 请把 RMSE 记在心里：**代理模型本身就有 1% 以上的误差**，而且这还是在和训练数据同分布的测试集上。第 4 节会看到，一旦离开这个分布，它会错得多离谱。

---
## 2. 微调

📘 让生成模型偏向高 PCE 分子，最简单的办法叫**微调**（fine-tuning），也叫迁移学习：

1. 先在**全部**数据上训练，让模型学会"怎么写一个合法的 CEP 分子"（这就是 08a 做的事）；
2. 再在**一小部分优质数据**上用**小学习率**继续训练几轮，把模型的分布往那边挪一点。

💡 还有更强的办法：**强化学习**（REINVENT 用的就是这个），把预测的 PCE 当作奖励，直接优化模型去最大化它。微调的好处是简单、稳定、不容易把模型搞崩。

▶️ 先把 08a 的分词、模型和采样代码搬过来（和 08a 完全相同）：

In [ ]:
TOKEN_PATTERN = re.compile(r"(\[[^\]]+\]|Br|Cl|[BCNOPSFIbcnops]|\(|\)|\.|=|#|-|\+|\\|/|:|%\d{2}|\d)")

def tokenize(smiles):
    tokens = TOKEN_PATTERN.findall(smiles)
    assert "".join(tokens) == smiles, smiles
    return tokens

token_lists = [tokenize(s) for s in df["smiles"]]
vocab = ["<pad>", "^", "$"] + sorted({t for ts in token_lists for t in ts})
stoi = {t: i for i, t in enumerate(vocab)}
max_len = max(len(ts) for ts in token_lists) + 2

data = torch.zeros(len(token_lists), max_len, dtype=torch.long)
for i, tokens in enumerate(token_lists):
    ids = [stoi["^"]] + [stoi[t] for t in tokens] + [stoi["$"]]
    data[i, :len(ids)] = torch.tensor(ids)
print("词表:", len(vocab), " 最长序列:", max_len)

In [ ]:
class SmilesRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_size=256, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)   # 每个 token -> 一个向量
        self.lstm = nn.LSTM(embedding_dim, hidden_size, n_layers, batch_first=True, dropout=0.1)
        self.linear = nn.Linear(hidden_size, vocab_size)           # 隐状态 -> 每个 token 的分数

    def forward(self, x, state=None):
        """x: (batch, 长度) 的 token 编号。返回每个位置上"下一个 token"的分数。"""
        out, state = self.lstm(self.embedding(x), state)
        return self.linear(out), state

In [ ]:
@torch.no_grad()
def sample(model, n=500, temperature=1.0, max_steps=max_len):
    """从模型里采样 n 个 SMILES 字符串。"""
    model.eval()
    x = torch.full((n, 1), stoi["^"])             # 所有序列都从 ^ 开始
    state = None
    out = torch.zeros(n, 0, dtype=torch.long)
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(max_steps):
        logits, state = model(x, state)            # 只喂最新的一个 token，state 记住了前面的内容
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)            # 按概率抽一个 token（不是取最大的！）
        out = torch.cat([out, x], dim=1)
        finished |= (x[:, 0] == stoi["$"])
        if finished.all():
            break
    model.train()

    smiles_list = []
    for row in out.tolist():
        tokens = []
        for t in row:
            if t == stoi["$"]:
                break
            tokens.append(vocab[t])
        smiles_list.append("".join(tokens))
    return smiles_list

In [ ]:
CKPT = os.path.join(DATA_DIR, "cep_rnn.pt")
if not os.path.exists(CKPT):
    raise FileNotFoundError("找不到 data/cep_rnn.pt。请先运行 05a，或把它和 cep.csv 一起上传。")

prior_model = SmilesRNN(len(vocab))                               # "先验模型"：05a 训练好的
prior_model.load_state_dict(torch.load(CKPT, map_location="cpu"))
prior_model.eval()
print("已加载预训练模型")

### 2.1 选出"优质数据"

📘 用 PCE 最高的 10% 作为微调数据集。

🤔 为什么不用最高的 1%？或者 50%？

In [ ]:
top_fraction = 0.10                                               # ✏️ 试试 0.02 或 0.3
n_top = int(len(df) * top_fraction)
top_idx = np.argsort(-y)[:n_top]
print(f"微调集: {n_top} 个分子, PCE 门槛 {y[top_idx].min():.2f}%, 平均 {y[top_idx].mean():.2f}%")

📘 这是个权衡：

- 取太少（比如 1%，约 300 个分子），模型很快就把这几百个**背下来**了，生成的分子会严重重复（唯一性暴跌）；
- 取太多（比如 50%），和原来的分布差别不大，微调基本没效果。

10% 是个常见的起点。练习 09b-1 会让你自己验证这个权衡。

### 2.2 微调

In [ ]:
tuned_model = copy.deepcopy(prior_model)                          # 从预训练模型出发
optimizer = torch.optim.Adam(tuned_model.parameters(), lr=1e-4)   # 学习率比预训练小 10 倍！
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

subset = data[top_idx]
tuned_model.train()
for epoch in range(4):
    order = np.random.permutation(len(subset))
    total = 0.0
    for i in range(0, len(order), 128):
        batch = subset[order[i:i + 128]]
        logits, _ = tuned_model(batch[:, :-1])
        loss = loss_fn(logits.reshape(-1, len(vocab)), batch[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(tuned_model.parameters(), 5)
        optimizer.step()
        total += loss.item()
    print(f"epoch {epoch}: loss {total / (len(order) / 128):.4f}")
tuned_model.eval();

---
## 3. 效果如何？

▶️ 两个模型各采样 2000 个分子，用代理模型给它们打分：

In [ ]:
canonical_train = {Chem.CanonSmiles(s) for s in df["smiles"]}

def analyse(model, name, n=2000, temperature=1.0):
    samples = sample(model, n, temperature=temperature)
    parsed = [Chem.MolFromSmiles(s) for s in samples]
    valid = [Chem.MolToSmiles(m) for m in parsed if m is not None and m.GetNumAtoms() > 0]
    unique = sorted(set(valid))
    fps = np.array([fp_gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(c)) for c in unique], dtype=np.uint8)
    predictions = rf.predict(fps)
    print(f"{name:10s} valid {len(valid) / n:.3f}  unique {len(unique) / max(len(valid), 1):.3f}  "
          f"novel {len(set(unique) - canonical_train) / max(len(unique), 1):.3f}  "
          f"预测 PCE 平均 {predictions.mean():.2f}%  最大 {predictions.max():.2f}%")
    return pd.DataFrame({"smiles": unique, "pred_PCE": predictions})

In [ ]:
before = analyse(prior_model, "微调前")
after = analyse(tuned_model, "微调后")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
bins = np.linspace(-2, 12, 50)
ax.hist(y, bins=bins, density=True, alpha=0.4, label="CEP training data (DFT)")
ax.hist(before["pred_PCE"], bins=bins, density=True, histtype="step", lw=2, label="generated before fine-tuning")
ax.hist(after["pred_PCE"], bins=bins, density=True, histtype="step", lw=2, label="generated after fine-tuning")
ax.axvline(y[top_idx].min(), color="gray", ls=":", label="top-10% threshold")
ax.set_xlabel("PCE (%)"); ax.set_ylabel("density"); ax.legend(fontsize=8)
ax.set_title("Does fine-tuning shift the distribution?")
plt.tight_layout(); plt.show()

📘 整条分布明显右移了。**微调起作用了。**

🤔 但注意：微调后的平均值并没有达到微调集的平均水平。为什么？

📘 几个原因叠加：

1. 只微调了 4 轮、学习率很小，模型只被"推"了一点点；
2. 模型要在"模仿高 PCE 分子"和"保持会写合法分子"之间平衡；
3. 代理模型本身有误差，而且随机森林的预测永远落在训练数据的范围内（02a 讲过），对新分子倾向于给出偏保守的分数。

▶️ 看看微调后预测 PCE 最高的几个分子：

In [ ]:
best = after.nlargest(8, "pred_PCE")
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in best["smiles"]], molsPerRow=4, subImgSize=(270, 210),
                     legends=[f"predicted {p:.2f}%" for p in best["pred_PCE"]])

🤔 这些分子有什么共同特点？和 08a 第 7.2 节那张"构筑单元"的图对照一下。

▶️ 统计一下微调前后，几个构筑单元出现频率的变化：

In [ ]:
fragments = {
    "thiadiazole": "c1nsnc1",
    "thiophene": "c1ccsc1",
    "furan": "c1ccoc1",
    "pyrrole": "c1cc[nH]c1",
    "benzene": "c1ccccc1",
    "selenophene": "c1cc[se]c1",
}

def fragment_frequency(smiles_list):
    parsed = [Chem.MolFromSmiles(s) for s in smiles_list]
    return {name: np.mean([m.HasSubstructMatch(Chem.MolFromSmiles(sm)) for m in parsed])
            for name, sm in fragments.items()}

freq = pd.DataFrame({
    "CEP all": fragment_frequency(df["smiles"].sample(2000, random_state=0)),
    "CEP top 10%": fragment_frequency(df["smiles"].values[top_idx][:2000]),
    "generated before": fragment_frequency(before["smiles"]),
    "generated after": fragment_frequency(after["smiles"]),
}).round(3)
freq

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
freq.plot.barh(ax=ax)
ax.set_xlabel("fraction of molecules containing the fragment")
ax.set_title("Fine-tuning shifts which building blocks appear")
plt.tight_layout(); plt.show()

📘 看 **噻二唑** 这一行：它在高 PCE 分子里出现得比在全体 CEP 分子里多，而微调后的生成分子也跟着变多了。

这很有意思：模型并不知道什么叫"受体单元"、什么叫"推拉电子结构"，它只是观察到"高 PCE 的分子里常有这个片段"，就跟着多写这个片段。而在有机光伏领域，**苯并噻二唑确实是最经典的吸电子单元之一**，窄带隙给体材料里到处都是它。

> 模型没有理解化学，但它复现了化学家总结出来的设计规则——因为这些规则本来就写在数据里。

---
## 4. ⚠️ 陷阱一：模型欺骗

📘 现在来说这一节最重要的内容。

我们在**优化一个代理模型的预测值**。代理模型只是真实 PCE 的一个近似，它在训练数据附近还准，离远了就不知道了。如果优化得太狠，生成模型会找到那些**让代理模型给出高分、但实际上根本不对**的分子。

这个现象有很多名字：model exploitation（模型欺骗）、reward hacking（奖励作弊）、对抗样本。它是定向分子生成**最常见的失败方式**。

### 4.1 用 03b 的办法诊断：它们在适用域内吗？

▶️ 对每个生成分子，算它到训练集的**最大 Tanimoto 相似度**。相似度越低，代理模型越不可信：

In [ ]:
reference = np.random.RandomState(0).choice(len(df), 3000, replace=False)
reference_fps = [fp_gen.GetFingerprint(mols[i]) for i in reference]

def max_similarity(smiles_list):
    out = []
    for s in smiles_list:
        fp = fp_gen.GetFingerprint(Chem.MolFromSmiles(s))
        out.append(max(DataStructs.BulkTanimotoSimilarity(fp, reference_fps)))
    return np.array(out)

sim_before = max_similarity(before["smiles"][:600])
sim_after = max_similarity(after["smiles"][:600])

# 参照组：训练集内部，分子彼此之间的最大相似度（排除它自己）
inside_sim = []
for s in df["smiles"].sample(600, random_state=1):
    fp = fp_gen.GetFingerprint(Chem.MolFromSmiles(s))
    sims = sorted(DataStructs.BulkTanimotoSimilarity(fp, reference_fps))
    inside_sim.append(sims[-2])
inside_sim = np.array(inside_sim)

print(f"训练集内部:   最大相似度中位数 {np.median(inside_sim):.3f}")
print(f"微调前生成:   {np.median(sim_before):.3f}")
print(f"微调后生成:   {np.median(sim_after):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(0.2, 1.0, 40)
ax.hist(inside_sim, bins=bins, density=True, alpha=0.5, label="CEP molecule vs CEP")
ax.hist(sim_after, bins=bins, density=True, histtype="step", lw=2, label="generated vs CEP")
ax.set_xlabel("max Tanimoto similarity to training set"); ax.set_ylabel("density"); ax.legend(fontsize=8)
ax.set_title("Are generated molecules inside the applicability domain?")
plt.tight_layout(); plt.show()

📘 生成分子到训练集的相似度分布，和训练分子彼此之间的分布很接近。微调后的模型没有跑到化学空间的荒原上去，代理模型的预测还算可信。

**这是微调相对于强化学习的一个优势**：微调模型始终被"写得像训练数据"这个目标牵着，不容易失控。而强化学习如果只优化奖励、不加约束，常常会生成一些极端分子。

### 4.2 拿荒唐的分子去问代理模型

📘 微调后的模型表现不错，但这是因为它**被训练数据牵着**。如果换一个不受约束的优化器（遗传算法、强化学习），它会跑到哪里去？

先做个更简单的实验：**直接拿一些明显荒唐的分子去问代理模型**，看它怎么回答。

🤔 先猜一猜：给随机森林一个**水分子**，它会说什么？

In [ ]:
benzothiadiazole = "c1ccc2nsnc2c1"
probes = {
    "benzothiadiazole ×1": benzothiadiazole,
    "benzothiadiazole ×3": "-".join([benzothiadiazole] * 3),
    "benzothiadiazole ×6": "-".join([benzothiadiazole] * 6),
    "benzothiadiazole ×12": "-".join([benzothiadiazole] * 12),
    "thiophene ×12": "-".join(["c1ccsc1"] * 12),
    "benzene ×20": "-".join(["c1ccccc1"] * 20),
    "n-octane（完全没有共轭）": "CCCCCCCC",
    "一长串烷烃 C40": "C" * 40,
    "水": "O",
}

rows = []
for name, smiles in probes.items():
    mol = Chem.MolFromSmiles(smiles)
    x = np.array([fp_gen.GetFingerprintAsNumPy(mol)], dtype=np.uint8)
    tree_preds = np.array([t.predict(x)[0] for t in rf.estimators_])
    rows.append({
        "molecule": name,
        "heavy atoms": mol.GetNumHeavyAtoms(),
        "predicted PCE": round(float(tree_preds.mean()), 2),
        "tree spread": round(float(tree_preds.std()), 2),
        "max similarity": round(max(DataStructs.BulkTanimotoSimilarity(fp_gen.GetFingerprint(mol), reference_fps)), 3),
    })

typical_spread = np.array([t.predict(fingerprints[test_idx[:300]]) for t in rf.estimators_]).std(axis=0).mean()
print(f"参照：测试集上树的分歧平均只有 {typical_spread:.2f}%，训练数据 PCE 平均 {y.mean():.2f}%、最大 {y.max():.2f}%")
print(f"      训练分子的重原子数范围: {min(m.GetNumHeavyAtoms() for m in mols)} – {max(m.GetNumHeavyAtoms() for m in mols)}")
pd.DataFrame(rows).set_index("molecule")

📘 这张表里有三个值得警惕的发现：

**1. 模型永远会给你一个数，哪怕问题毫无意义。**
一个**水分子**被预测有约 1.2% 的光电转换效率。模型不会说"这不是我该回答的问题"——它只会照常输出一个数字。n-octane、长链烷烃也一样，它们连 π 共轭体系都没有。

**2. 树的分歧不是可靠的"我不知道"信号。**
水和烷烃的树分歧（约 0.5%）**比测试集上的平均水平还低**！因为这些分子的指纹几乎是全零，所有的树都走到同一个叶子上，于是"一致地给出一个错误答案"。

> 集成模型的分歧小，只说明**各棵树看法一致**，不说明**它们是对的**。

**3. 相似度才是真正有用的警报。**
水的最大 Tanimoto 相似度是 **0.000**，烷烃是 0.03，而训练分子彼此之间的中位数约 0.49。这个信号干净利落地把它们标了出来。

🤔 再看 benzothiadiazole ×6 和 ×12 这两行：一个 54 个重原子，一个 108 个（训练数据最多只有 35 个），预测值却**一模一样**。为什么？

📘 因为 **Morgan 指纹是二值的**：第 7 个苯并噻二唑单元产生的子结构，前 6 个已经全部产生过了，对应的比特早就是 1，再多也不会变。**模型根本"看不见"分子变长了。**

这正是 01d 讲二值指纹 vs 计数指纹、02a 比较两者、10a 讨论并四苯和并五苯时反复出现的同一个问题。在生成模型的场景下它变得更危险：优化器会**专门去找**这种模型看不出区别的方向。

💡 **教训**：代理模型只在它见过的化学空间里有意义。把它交给一个不受约束的优化器，就等于邀请对方去找它的盲区。防止的办法：

1. **约束在适用域内**：相似度低于阈值的分子直接丢掉（最有效的一招）；
2. **限制搜索空间**：像微调那样，让生成模型始终"像训练数据"；
3. **主动学习**：把生成的分子拿去真的算 DFT，用新数据更新代理模型，再生成。这是现在最主流的闭环做法。

---
## 5. ⚠️ 陷阱二：能不能合成？

📘 就算预测的 PCE 真的很高、分子也在适用域里，还有最后一个问题：**这个分子做得出来吗？**

RDKit 自带一个粗略的**合成可及性评分** SA score（1 = 很容易，10 = 很难）：

In [ ]:
import sys
from rdkit.Chem import RDConfig
sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
import sascorer

def sa_scores(smiles_list):
    return np.array([sascorer.calculateScore(Chem.MolFromSmiles(s)) for s in smiles_list])

sa_train = sa_scores(df["smiles"].sample(1000, random_state=0))
sa_after = sa_scores(after["smiles"][:1000])
sa_best = sa_scores(best["smiles"])

fig, ax = plt.subplots(figsize=(6, 3.5))
bins = np.linspace(1, 8, 40)
ax.hist(sa_train, bins=bins, density=True, alpha=0.5, label="CEP training data")
ax.hist(sa_after, bins=bins, density=True, histtype="step", lw=2, label="generated (fine-tuned)")
ax.set_xlabel("SA score (1 = easy, 10 = hard)"); ax.set_ylabel("density"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"SA score 中位数: CEP {np.median(sa_train):.2f}, 微调后生成 {np.median(sa_after):.2f}, "
      f"预测 PCE 最高的 8 个 {np.median(sa_best):.2f}")

📘 微调生成的分子，合成难度和训练数据差不多。

⚠️ 但要小心：**CEP 数据集本身就是计算机枚举出来的**，里面很多分子从来没有人合成过。"和 CEP 一样难合成"并不等于"容易合成"。

💡 SA score 也只是一个粗糙的经验公式。真正判断可合成性要靠**逆合成分析**（ai4chem 第 8 章的主题），或者直接问合成化学家。

---
## 6. 一个诚实的检查清单

📘 拿到一批生成的候选分子之后，在告诉别人"我们设计了新材料"之前，至少要问自己：

| 问题 | 怎么查 | 本课哪里讲过 |
|---|---|---|
| 分子合法吗？ | RDKit 能不能解析 | 08a |
| 和训练数据像吗？ | 最大 Tanimoto 相似度 | 03b、本节第 4 节 |
| 代理模型有把握吗？ | 集成模型内部的预测分歧 | 本节第 4 节 |
| 性质分布合理吗？ | 和训练集比描述符分布 | 08a 第 7 节、09a 第 6 节 |
| 做得出来吗？ | SA score、逆合成 | 本节第 5 节 |
| 真的好吗？ | **拿去做真实的计算或实验** | —— |

最后一条才是决定性的。**生成模型的产物是"值得一试的候选"，不是"结论"**——这和 03c 钯二聚体那个案例的结论完全一样：机器学习缩小搜索范围，实验给出答案。

### 🏋️ 练习 09b-1

第 2.1 节说微调集太小会导致模型"背书"。验证它：分别用 PCE 最高的 1%、10%、30% 做微调集（各 4 轮，其余设置不变），每次采样 1000 个分子，比较：

1. 唯一性（unique）；
2. 新颖性（novel）；
3. 预测 PCE 的平均值。

把三组结果列成一张表。你会看到什么样的权衡？

提示：每次都要从 `prior_model` 重新 `copy.deepcopy` 一份，不要在已经微调过的模型上接着调。

### 🏋️ 练习 09b-2

第 4.2 节是"手动"探针。现在让一个**优化器**自己去找。我们用最简单的分子表示：把若干个芳香单元用 `-` 串成一条链。

```python
FRAGMENTS = ["c1ccsc1", "c1ccoc1", "c1cc[nH]c1", "c1ccccc1",
             "c1cc[se]c1", "c1ccc2nsnc2c1", "c1ccc2[nH]ccc2c1"]
chain = ["c1ccsc1", "c1ccc2nsnc2c1", "c1ccsc1"]
smiles = "-".join(chain)          # 'c1ccsc1-c1ccc2nsnc2c1-c1ccsc1'
```

实现一个爬山法：每一步随机**插入 / 删除 / 替换**一个片段，用代理模型打分，更高就接受，跑 150 步。做两个版本：

1. **无约束**：目标函数就是预测的 PCE；
2. **有约束**：候选分子到训练集的最大相似度 < 0.4 就直接拒绝，并且目标函数改成"预测 PCE − 2 × 树的标准差"（悲观估计）。

比较两者终点的：预测 PCE、到训练集的相似度、树的分歧、重原子数。画出两个分子。哪一个你更愿意拿去做 DFT 验证？

### 🏋️ 练习 09b-3

**检验代理模型到底可不可信。** 我们没法做 DFT，但可以做一个"留出检验"：

1. 重新训练一个随机森林，这次**只用 PCE < 6% 的分子**；
2. 用它预测 PCE > 8% 的那些分子（它从没见过这么好的分子）；
3. 画预测值 vs 真实值，算出平均低估了多少。

它能发现那些高 PCE 分子吗？这个实验说明了什么？（提示：想想 02a 的随机森林为什么把并五苯预测得和并四苯一样。）

In [ ]:
# 在这里写你的代码

---
## 小结：第五部分结束

| 学到了 | 要点 |
|---|---|
| 代理模型 | 用快速模型代替昂贵的 DFT 当裁判；它本身有误差 |
| 微调 | 预训练 + 小学习率在优质子集上继续训练；简单稳定 |
| 效果 | 预测 PCE 分布明显右移；噻二唑等受体单元出现频率上升 |
| 模型欺骗 | 优化代理模型的缺陷而不是真实性质；用相似度和集成分歧诊断 |
| 可合成性 | SA score 只是粗略参考；枚举数据集本身也未必可合成 |
| 检查清单 | 合法、相似、有把握、分布合理、做得出、真的好 |

第五部分的三个 notebook：

- **08a**：SMILES 语言模型。生成模型学的是训练数据的分布——生成分子的描述符分布和构筑单元频率都和 CEP 几乎一致。
- **09a**：SELFIES + VAE。潜空间可以插值，但"先验空洞"让随机采样生成了一堆合法却不合理的分子。valid/unique/novel 三个指标会骗人。
- **09b**：微调把分布推向高 PCE，模型"重新发现"了噻二唑这类受体单元；同时也看到了模型欺骗这个最常见的陷阱。

**再往后（ai4chem 的后续章节）**：反应预测、逆合成、贝叶斯优化。有了候选分子之后，下一个问题自然就是——**这个分子该怎么合成？**